In [19]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.utils import to_categorical

In [20]:
df = pd.read_csv("test.csv")      
print(df.head())

   Class Index                                              Title  \
0            3                  Fears for T N pension after talks   
1            4  The Race is On: Second Private Team Sets Launc...   
2            4      Ky. Company Wins Grant to Study Peptides (AP)   
3            4      Prediction Unit Helps Forecast Wildfires (AP)   
4            4        Calif. Aims to Limit Farm-Related Smog (AP)   

                                         Description  
0  Unions representing workers at Turner   Newall...  
1  SPACE.com - TORONTO, Canada -- A second\team o...  
2  AP - A company founded by a chemistry research...  
3  AP - It's barely dawn when Mike Fitzpatrick st...  
4  AP - Southern California's smog-fighting agenc...  


In [21]:
df["text"] = df["Title"] + " " + df["Description"]

In [22]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

df["text"] = df["text"].apply(clean_text)


In [23]:
X = df["text"]

In [24]:
y = df["Class Index"] - 1

In [25]:
max_words = 10000

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X)

X_seq = tokenizer.texts_to_sequences(X)


In [26]:
max_length = 100

X_pad = pad_sequences(X_seq,
                      maxlen=max_length,
                      padding='post',
                      truncating='post')


In [27]:
y = to_categorical(y, num_classes=4)

In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    X_pad,
    y,
    test_size=0.2,
    random_state=42
)

In [30]:
model = Sequential()

model.add(Embedding(input_dim=max_words,
                    output_dim=128,
                    input_length=max_length))

model.add(SimpleRNN(64))

model.add(Dense(32, activation='relu'))

model.add(Dense(4, activation='softmax'))


In [31]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [32]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/5
152/152 ━━━━━━━━━━━━━━━━━━━━ 65s 150ms/step - accuracy: 0.2580 - loss: 1.3933 - val_accuracy: 0.2508 - val_loss: 1.3900
Epoch 2/5
152/152 ━━━━━━━━━━━━━━━━━━━━ 36s 140ms/step - accuracy: 0.2763 - loss: 1.3876 - val_accuracy: 0.2368 - val_loss: 1.4244
Epoch 3/5
152/152 ━━━━━━━━━━━━━━━━━━━━ 41s 138ms/step - accuracy: 0.2810 - loss: 1.3767 - val_accuracy: 0.2821 - val_loss: 1.3829
Epoch 4/5
152/152 ━━━━━━━━━━━━━━━━━━━━ 41s 133ms/step - accuracy: 0.2829 - loss: 1.3759 - val_accuracy: 0.2796 - val_loss: 1.3845
Epoch 5/5
152/152 ━━━━━━━━━━━━━━━━━━━━ 19s 120ms/step - accuracy: 0.2944 - loss: 1.3711 - val_accuracy: 0.2804 - val_loss: 1.3776


In [33]:
loss, accuracy = model.evaluate(X_test, y_test)
print("\nTest Accuracy:", accuracy)

48/48 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - accuracy: 0.2757 - loss: 1.3811

Test Accuracy: 0.27565789222717285


In [35]:
model.save("ag_news_rnn.h5")

In [40]:
news = """
Apple launches a new AI-powered iPhone with advanced features.
"""

news = clean_text(news)

seq = tokenizer.texts_to_sequences([news])

pad = pad_sequences(seq,
                    maxlen=max_length,
                    padding='post')

prediction = model.predict(pad)
label = np.argmax(prediction)
classes = {
    1: "World",
    2: "Sports",
    3: "Business",
    4: "Science/Technology"
}

predicted_index = label + 1

print("Predicted Class Index:", predicted_index)
print("Predicted Category:", classes[predicted_index])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 197ms/step
Predicted Class Index: 1
Predicted Category: World
